In [1]:

from pathlib import Path
import numpy as np
import pandas as pd


In [2]:

# ============================================================
# 1. CONFIGURATION
# ============================================================

# Project root (one level above notebooks/)
BS_DIR = Path("..")

# Input files
in_files = [
    BS_DIR / "data/Raw/accepted_2007_to_2018Q4.csv",
    BS_DIR / "data/Raw/rejected_2007_to_2018Q4.csv",
]

# Output directory
OUTPUT_DIR = BS_DIR / "data/Sampled"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Reproducibility settings
SEED = 69
CHUNK_SIZE = 100_000

# Three mutually exclusive samples per input file
SAMPLE_SIZES = {
    "sample": 50_000,
    "validation_sample": 50_000,
    "scaling_sample": 200_000,
}

# ============================================================
# 2. DETERMINISTIC RANDOM RANKING
# ============================================================

def stable_scores(n_rows, seed):
    """
    Generate deterministic 64-bit scores for row positions.
    Independent of pandas/NumPy random-number generators.
    """
    x = np.arange(n_rows, dtype=np.uint64)
    x ^= np.uint64(seed)

    with np.errstate(over="ignore"):
        x += np.uint64(0x9E3779B97F4A7C15)
        x = (x ^ (x >> np.uint64(30))) * np.uint64(
            0xBF58476D1CE4E5B9
        )
        x = (x ^ (x >> np.uint64(27))) * np.uint64(
            0x94D049BB133111EB
        )
        x ^= x >> np.uint64(31)

    return x


def generate_samples(input_path):
    print(f"\nProcessing: {input_path.name}")

    # --------------------------------------------------------
    # Pass 1: Count parsed CSV records without loading all data
    # --------------------------------------------------------
    n_rows = 0

    for chunk in pd.read_csv(
        input_path,
        usecols=[0],
        chunksize=CHUNK_SIZE,
    ):
        n_rows += len(chunk)

    print(f"Total rows: {n_rows:,}")

    total_needed = sum(SAMPLE_SIZES.values())

    if n_rows < total_needed:
        raise ValueError(
            f"{input_path.name} contains {n_rows:,} rows, "
            f"but {total_needed:,} are required."
        )

    # --------------------------------------------------------
    # Pass 2: Rank row positions deterministically
    # --------------------------------------------------------
    scores = stable_scores(n_rows, SEED)

    # Select the 300K lowest scores, then rank those rows.
    selected = np.argpartition(scores, total_needed)[
        :total_needed
    ]
    selected = selected[np.argsort(scores[selected])]

    # Assign separate, non-overlapping groups.
    sample_positions = {}
    start = 0

    for sample_name, size in SAMPLE_SIZES.items():
        positions = selected[start:start + size]

        # Restore original file order.
        sample_positions[sample_name] = np.sort(positions)
        start += size

    del scores, selected

    # --------------------------------------------------------
    # Pass 3: Read the full file in chunks and write samples
    # --------------------------------------------------------
    output_paths = {
        sample_name: OUTPUT_DIR / (
            f"{sample_name}_{input_path.stem}.csv"
        )
        for sample_name in SAMPLE_SIZES
    }

    # Remove previous outputs so reruns cannot append duplicates.
    for path in output_paths.values():
        if path.exists():
            path.unlink()

    written = {name: 0 for name in SAMPLE_SIZES}
    row_offset = 0

    for chunk in pd.read_csv(
        input_path,
        chunksize=CHUNK_SIZE,
    ):
        chunk_end = row_offset + len(chunk)

        for sample_name, positions in sample_positions.items():
            left = np.searchsorted(positions, row_offset)
            right = np.searchsorted(positions, chunk_end)

            if left == right:
                continue

            # Convert original row positions to chunk positions.
            local_positions = (
                positions[left:right] - row_offset
            )
            sample = chunk.iloc[local_positions]

            sample.to_csv(
                output_paths[sample_name],
                mode="a",
                header=(written[sample_name] == 0),
                index=False,
            )

            written[sample_name] += len(sample)

        row_offset = chunk_end

    # --------------------------------------------------------
    # Pass 4: Report results
    # --------------------------------------------------------
    for sample_name, output_path in output_paths.items():
        print(
            f"{sample_name}: {written[sample_name]:,} rows "
            f"-> {output_path}"
        )


# ============================================================
# 3. GENERATE SAMPLES FOR BOTH FILES
# ============================================================

for input_path in in_files:
    if not input_path.exists():
        raise FileNotFoundError(
            f"Input file not found: {input_path.resolve()}"
        )

    generate_samples(input_path)

print("\nAll samples generated successfully.")



Processing: accepted_2007_to_2018Q4.csv


C:\Users\M3\AppData\Local\Temp\ipykernel_22224\3820436393.py:62: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\M3\AppData\Local\Temp\ipykernel_22224\3820436393.py:62: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\M3\AppData\Local\Temp\ipykernel_22224\3820436393.py:62: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\M3\AppData\Local\Temp\ipykernel_22224\3820436393.py:62: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\M3\AppData\Local\Temp\ipykernel_22224\3820436393.py:62: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\M3\AppData\Local\Temp\ipyk

Total rows: 2,260,701


C:\Users\M3\AppData\Local\Temp\ipykernel_22224\3820436393.py:121: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\M3\AppData\Local\Temp\ipykernel_22224\3820436393.py:121: DtypeWarning: Columns (19,59) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\M3\AppData\Local\Temp\ipykernel_22224\3820436393.py:121: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\M3\AppData\Local\Temp\ipykernel_22224\3820436393.py:121: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\M3\AppData\Local\Temp\ipykernel_22224\3820436393.py:121: DtypeWarning: Columns (0,19,59,118) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\M3\A

sample: 50,000 rows -> ..\data\Sampled\sample_accepted_2007_to_2018Q4.csv
validation_sample: 50,000 rows -> ..\data\Sampled\validation_sample_accepted_2007_to_2018Q4.csv
scaling_sample: 200,000 rows -> ..\data\Sampled\scaling_sample_accepted_2007_to_2018Q4.csv

Processing: rejected_2007_to_2018Q4.csv
Total rows: 27,648,741
sample: 50,000 rows -> ..\data\Sampled\sample_rejected_2007_to_2018Q4.csv
validation_sample: 50,000 rows -> ..\data\Sampled\validation_sample_rejected_2007_to_2018Q4.csv
scaling_sample: 200,000 rows -> ..\data\Sampled\scaling_sample_rejected_2007_to_2018Q4.csv

All samples generated successfully.


In [18]:
import pandas as pd
from pathlib import Path

In [ ]:

# Project root (one level above notebooks/)
BS_DIR = Path("..")

# Available input files
in_files = [
    BS_DIR / "data/Raw/accepted_2007_to_2018Q4.csv",
    BS_DIR / "data/Raw/rejected_2007_to_2018Q4.csv"
]

# Available output files
out_files = [
    BS_DIR / "data/Sampled/sample_accepted_2007_to_2018Q4.csv",
    BS_DIR / "data/Sampled/sample_rejected_2007_to_2018Q4.csv",
    BS_DIR / "data/Sampled/validation_sample_accepted_2007_to_2018Q4.csv",
    BS_DIR / "data/Sampled/validation_sample_rejected_2007_to_2018Q4.csv"
    BS_DIR / "data/Sampled/scaling_sample_accepted_2007_to_2018Q4.csv",
    BS_DIR / "data/Sampled/scaling_sample_rejected_2007_to_2018Q4.csv"
]

# # Select file (0 = accepted, 1 = rejected)
# file_index = 0

input_target = in_files[0]
output_target = out_files[0]

# Check input file size
size_gb = input_target.stat().st_size / (1024 ** 3)
print(f"Input file size: {size_gb:.2f} GiB")


Input file size: 1.56 GiB


In [20]:

# 1. Read only the column names
columns = pd.read_csv(
    input_target,
    nrows=0
).columns.tolist()

print(f"\nNumber of columns: {len(columns)}")

# 2. Display the column names
for index, column in enumerate(columns, start=1):
    print(f"{index:3}. {column}")

# 3. Read a small sample
sample = pd.read_csv(
    input_target,
    # nrows=10,
    low_memory=False
)


Number of columns: 151
  1. id
  2. member_id
  3. loan_amnt
  4. funded_amnt
  5. funded_amnt_inv
  6. term
  7. int_rate
  8. installment
  9. grade
 10. sub_grade
 11. emp_title
 12. emp_length
 13. home_ownership
 14. annual_inc
 15. verification_status
 16. issue_d
 17. loan_status
 18. pymnt_plan
 19. url
 20. desc
 21. purpose
 22. title
 23. zip_code
 24. addr_state
 25. dti
 26. delinq_2yrs
 27. earliest_cr_line
 28. fico_range_low
 29. fico_range_high
 30. inq_last_6mths
 31. mths_since_last_delinq
 32. mths_since_last_record
 33. open_acc
 34. pub_rec
 35. revol_bal
 36. revol_util
 37. total_acc
 38. initial_list_status
 39. out_prncp
 40. out_prncp_inv
 41. total_pymnt
 42. total_pymnt_inv
 43. total_rec_prncp
 44. total_rec_int
 45. total_rec_late_fee
 46. recoveries
 47. collection_recovery_fee
 48. last_pymnt_d
 49. last_pymnt_amnt
 50. next_pymnt_d
 51. last_credit_pull_d
 52. last_fico_range_high
 53. last_fico_range_low
 54. collections_12_mths_ex_med
 55. mths_sinc

In [21]:
sample

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2260696,88985880,NaN,40000.0,40000.0,40000.0,60 months,10.49,859.56,B,B3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2260697,88224441,NaN,24000.0,24000.0,24000.0,60 months,14.49,564.56,C,C4,...,NaN,NaN,Cash,Y,Mar-2019,ACTIVE,Mar-2019,10000.0,44.82,1.0
2260698,88215728,NaN,14000.0,14000.0,14000.0,60 months,14.49,329.33,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2260699,Total amount funded in policy code 1: 1465324575,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
